<a href="https://colab.research.google.com/github/sreesreevallichowdary-png/SQL-Joining-Data/blob/main/Assignment_SQL_Joining_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#Task 1: Basic Joins (INNER and LEFT)
import pandas as pd
import sqlite3
import seaborn as sns
# Create database
conn = sqlite3.connect('joins_practice.db')

# Table Schemas:
students = pd.DataFrame({
    'student_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Carol', 'David', 'Emma'],
    'email':['alice@email.com','bob@email.com','carol@email.com','david@email.com','emma@email.com'],
    'city':['Mumbai','Delhi','Bangalore','Chennai','Pune']
})

enrollments = pd.DataFrame({
    'enrollment_id':['101','102','103','104','105'],
    'student_id': [1, 1,2,3,5],
    'course_name': ['Python', 'SQL', 'Data Science','Python','Machine Learning'],
    'enrollment_date': ['2024-01-15', '2024-02-01', '2024-01-20','2024-02-10','2024-01-25']
})

grades = pd.DataFrame({
    'grade_id': [201, 202, 203,204],
    'student_id': [1,1,2,4],
    'course_name': ['Python', 'SQL', 'Data Science','Web Development'],
    'score': [85,90,78,88]
})

# Save to database
students.to_sql('students', conn, if_exists='replace', index=False)
enrollments.to_sql('enrollments', conn, if_exists='replace', index=False)
grades.to_sql('grades', conn, if_exists='replace', index=False)

#Tak1: INNER JOIN: Get student names and their enrolled courses
#1):INNER JOIN: Get student names and their enrolled courses
query = '''
SELECT name,course_name

FROM students INNER JOIN enrollments on students.student_id = enrollments.student_id '''
df_sql = pd.read_sql(query, conn)
print(df_sql)

#Explain what data this returns and which students appear.
#- INNER JOIN only returns rows where there is a match in both tables.
#- That means students who are not enrolled in any course will be excluded entirely.

#2)LEFT JOIN: Get all students and their enrolled courses (including students with no enrollments)

query = '''
SELECT name,course_name

FROM students LEFT JOIN enrollments on students.student_id = enrollments.student_id '''
df_sql = pd.read_sql(query, conn)
print(df_sql)

conn.close()


#- It performs a LEFT JOIN between students and enrollments.
#- That means all students will appear in the result, even if they don’t have any enrollments.
#- For students who do have enrollments, their course_name will be shown.
#- For students without enrollments, the course_name will be NULL.

#3)Why do we use table aliases like AS s and AS e? Rewrite Query 1 without aliases and explain why aliases are helpful.

#Aliases (like AS s for students and AS e for enrollments) are essentially shortcuts. They don’t change the logic of the query, but they make it easier to read and write, especially when:
#- Tables have long names
#Instead of typing students.student_id repeatedly, you can just write s.student_id.
#- Multiple joins are involved
#When you join 3–4 tables, writing full names gets messy. Aliases keep queries concise and clear.
#- Self-joins
#If you join a table to itself (e.g., employees referencing managers in the same table), aliases are the only way to distinguish between the two instances.
#- Readability in SELECT clause
#alias is easier than full df name.




    name       course_name
0  Alice            Python
1  Alice               SQL
2    Bob      Data Science
3  Carol            Python
4   Emma  Machine Learning
    name       course_name
0  Alice            Python
1  Alice               SQL
2    Bob      Data Science
3  Carol            Python
4  David              None
5   Emma  Machine Learning


In [8]:
#Task 2: Multiple Table Joins
#Get student name, enrolled course, and score for each enrollment:

conn = sqlite3.connect('joins_practice.db')

query = """
SELECT
    students.name,
    enrollments.course_name,
    grades.score

FROM students
LEFT JOIN enrollments
    ON students.student_id = enrollments.student_id
LEFT JOIN grades
    ON students.student_id = grades.student_id
ORDER BY enrollments.enrollment_id
"""

df_sql = pd.read_sql(query, conn)
print(df_sql)

conn.close()

#2)
#Question1. Why we use LEFT JOIN instead of INNER JOIN here
    #- In short: LEFT JOIN preserves the full student list and fills in missing values with NULL.
    #- If you used an INNER JOIN, only students who have both an enrollment and a grade would show up. That would exclude David (who has a grade but no enrollment) and Emma/Carol (who have enrollments but no grades).

#Question2. What happens to students with no enrollments
    #- They still appear in the result because of the LEFT JOIN with enrollments.
    #- Their course_name will be NULL.

#Question3. What happens to enrollments with no grades
    #- Their score will be NULL.
    #- Example: Carol (Python) and Emma (Machine Learning) have enrollments but no grades, so their scores are NaN.

#Why the join condition includes both student_id AND course_name
    #- In current query, only joined on student_id. That means all grades for a student are matched to all enrollments for that student, even if the course names don’t align.
    #- That’s why Alice’s Python enrollment shows both her Python score (85) and her SQL score (90), and her SQL enrollment also shows both scores.
    #- To avoid this duplication and ensure the grade matches the correct course, typically we use join on both student_id and course_name:









    name       course_name  score
0  David              None   88.0
1  Alice            Python   85.0
2  Alice            Python   90.0
3  Alice               SQL   85.0
4  Alice               SQL   90.0
5    Bob      Data Science   78.0
6  Carol            Python    NaN
7   Emma  Machine Learning    NaN


In [11]:
#Task 3: Understanding Joins and Normalization

#1. Why normalize data? Explain why student information is split across three tables instead of one large table with all data repeated.

    #Normalization is the process of organizing data into separate tables to:
    #- Avoid redundancy: If you kept all student info, enrollments, and grades in one giant table, you’d repeat the student’s name, email, and city for every course they take.
    #- Ensure consistency: If Alice’s email changes, you only update it once in the students table, instead of hunting through dozens of rows in a big combined table.
    #- Improve efficiency: Smaller, focused tables are faster to query and easier to maintain.

#2. When to use each join type:

#Use INNER JOIN when:
    #- You only want rows where there is a match in both tables.
    #- Result: Students without enrollments are excluded.

#Use LEFT JOIN when:
    #- You want all rows from the left table, regardless of whether there’s a match in the right table.
    #- Result: Students with no enrollments appear, with NULL values for course fields

#3. Pandas equivalent: If you had these three tables as DataFrames (students_df, enrollments_df, grades_df), write the Pandas code to perform the same multiple join from Task 2:

import pandas as pd

# First join: students with enrollments (LEFT JOIN)
merged = pd.merge(
    students,
    enrollments,
    on="student_id",
    how="left"
)

# Second join: add grades (LEFT JOIN, match on both student_id and course_name)
final = pd.merge(
    merged,
    grades,
    on=["student_id", "course_name"],
    how="left"
)

# Select only the relevant columns
result = final[["name", "course_name", "score"]]

print(result)



#- First merge:
#- students_df with enrollments_df on student_id.
#- how="left" ensures all students appear, even if they have no enrollments.
#- Second merge:
#- The merged DataFrame with grades_df.
#- Join on both student_id and course_name to avoid mismatched scores.
#- Again how="left" so enrollments without grades still appear (with NaN for score).
#- Final selection:
#- Keep only name, course_name, and score for clarity.








    name       course_name  score
0  Alice            Python   85.0
1  Alice               SQL   90.0
2    Bob      Data Science   78.0
3  Carol            Python    NaN
4  David               NaN    NaN
5   Emma  Machine Learning    NaN
